### Overview
Making a prediction using a linear regression model is a common use case in ML. In this guide tutorial, we build the model that predicts if a driver will complete a trip based on a number of features ingested into Feast.

The basic local mode gives you ability to quickly try Feast, while the advanced mode shows how you can use Feast in a production setting, in particular for the Google Cloud Platform (GCP) cloud.

This tutorial uses Feast with scikit learn to:

* Train a model locally using data from BigQuery
* Test the model for online inference using SQLite (for fast iteration)
* Test the model for online inference using Firestore (to represent production)


## Step 1: Install feast, scikit-learn

Install feast, gcp dependencies and scikit-learn


In [1]:
!pip install feast scikit-learn 'feast[gcp]'

INFO: pip is looking at multiple versions of uvicorn-worker to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of grpcio-status to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.3/8.3 MB 48.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.7/131.7 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 567.2/567.2 kB 16.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.8/131.8 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.1/64.1 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 327.1/327.1 kB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 17.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.3/62.3 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.0/212.0 kB 9.0 MB

#### Check feast version

In [10]:
!feast version

/usr/local/lib/python3.12/dist-packages/matplotlib/_fontconfig_pattern.py:64: PyparsingDeprecationWarning: 'oneOf' deprecated - use 'one_of'
  prop = Group((name + Suppress("=") + comma_separated(value)) | oneOf(_CONSTANTS))
/usr/local/lib/python3.12/dist-packages/matplotlib/_fontconfig_pattern.py:85: PyparsingDeprecationWarning: 'parseString' deprecated - use 'parse_string'
  parse = parser.parseString(pattern)
/usr/local/lib/python3.12/dist-packages/matplotlib/_fontconfig_pattern.py:89: PyparsingDeprecationWarning: 'resetCache' deprecated - use 'reset_cache'
  parser.resetCache()
/usr/local/lib/python3.12/dist-packages/matplotlib/_mathtext.py:45: PyparsingDeprecationWarning: 'enablePackrat' deprecated - use 'enable_packrat'
  ParserElement.enablePackrat()
In /usr/local/lib/python3.12/dist-packages/matplotlib/mpl-data/stylelib/classic.mplstyle: 'parseString' deprecated - use 'parse_string'
In /usr/local/lib/python3.12/dist-packages/matplotlib/mpl-data/stylelib/classic.mplstyle: 'reset

In [3]:
! feast init iris-feature-store-creation-sqlite

/usr/local/lib/python3.12/dist-packages/matplotlib/_fontconfig_pattern.py:64: PyparsingDeprecationWarning: 'oneOf' deprecated - use 'one_of'
  prop = Group((name + Suppress("=") + comma_separated(value)) | oneOf(_CONSTANTS))
/usr/local/lib/python3.12/dist-packages/matplotlib/_fontconfig_pattern.py:85: PyparsingDeprecationWarning: 'parseString' deprecated - use 'parse_string'
  parse = parser.parseString(pattern)
/usr/local/lib/python3.12/dist-packages/matplotlib/_fontconfig_pattern.py:89: PyparsingDeprecationWarning: 'resetCache' deprecated - use 'reset_cache'
  parser.resetCache()
/usr/local/lib/python3.12/dist-packages/matplotlib/_mathtext.py:45: PyparsingDeprecationWarning: 'enablePackrat' deprecated - use 'enable_packrat'
  ParserElement.enablePackrat()
In /usr/local/lib/python3.12/dist-packages/matplotlib/mpl-data/stylelib/classic.mplstyle: 'parseString' deprecated - use 'parse_string'
In /usr/local/lib/python3.12/dist-packages/matplotlib/mpl-data/stylelib/classic.mplstyle: 'reset

In [4]:
!pwd

/content


In [8]:
import pandas as pd

df = pd.read_csv("/content/iris_data_adapted_for_feast.csv")

df["sepal_area"] = df["sepal_length"] * df["sepal_width"]
df["petal_area"] = df["petal_length"] * df["petal_width"]
df["sepal_ratio"] = df["sepal_length"] / df["sepal_width"]
df["petal_ratio"] = df["petal_length"] / df["petal_width"]
df["flower_area"] = df["sepal_area"] + df["petal_area"]
df["petal_to_sepal_ratio"] = df["petal_area"] / df["sepal_area"]

df["event_timestamp"] = pd.to_datetime(
    df["event_timestamp"],
    utc=True
)

df["created_timestamp"] = pd.to_datetime(
    df["created_timestamp"],
    utc=True
)

df.to_parquet("/content//iris-feature-store-creation-sqlite/feature_repo/data/iris.parquet", index=False)

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [10]:
%cd /content/iris-feature-store-creation-sqlite/feature_repo
!feast apply

/content/iris-feature-store-creation-sqlite/feature_repo
/usr/local/lib/python3.12/dist-packages/matplotlib/_fontconfig_pattern.py:64: PyparsingDeprecationWarning: 'oneOf' deprecated - use 'one_of'
  prop = Group((name + Suppress("=") + comma_separated(value)) | oneOf(_CONSTANTS))
/usr/local/lib/python3.12/dist-packages/matplotlib/_fontconfig_pattern.py:85: PyparsingDeprecationWarning: 'parseString' deprecated - use 'parse_string'
  parse = parser.parseString(pattern)
/usr/local/lib/python3.12/dist-packages/matplotlib/_fontconfig_pattern.py:89: PyparsingDeprecationWarning: 'resetCache' deprecated - use 'reset_cache'
  parser.resetCache()
/usr/local/lib/python3.12/dist-packages/matplotlib/_mathtext.py:45: PyparsingDeprecationWarning: 'enablePackrat' deprecated - use 'enable_packrat'
  ParserElement.enablePackrat()
In /usr/local/lib/python3.12/dist-packages/matplotlib/mpl-data/stylelib/classic.mplstyle: 'parseString' deprecated - use 'parse_string'
In /usr/local/lib/python3.12/dist-packa

In [12]:
!pwd

/content


In [11]:
%cd /content

/content


In [24]:
!zip -r iris-feature-store-creation-sqlite.zip iris-feature-store-creation-sqlite

  adding: iris-feature-store-creation-sqlite/ (stored 0%)
  adding: iris-feature-store-creation-sqlite/__init__.py (stored 0%)
  adding: iris-feature-store-creation-sqlite/.gitignore (deflated 35%)
  adding: iris-feature-store-creation-sqlite/feature_repo/ (stored 0%)
  adding: iris-feature-store-creation-sqlite/feature_repo/feature_definitions.py (deflated 77%)
  adding: iris-feature-store-creation-sqlite/feature_repo/__init__.py (stored 0%)
  adding: iris-feature-store-creation-sqlite/feature_repo/__pycache__/ (stored 0%)
  adding: iris-feature-store-creation-sqlite/feature_repo/__pycache__/test_workflow.cpython-312.pyc (deflated 48%)
  adding: iris-feature-store-creation-sqlite/feature_repo/__pycache__/__init__.cpython-312.pyc (deflated 20%)
  adding: iris-feature-store-creation-sqlite/feature_repo/__pycache__/feature_definitions.cpython-312.pyc (deflated 47%)
  adding: iris-feature-store-creation-sqlite/feature_repo/data/ (stored 0%)
  adding: iris-feature-store-creation-sqlite/fea

In [25]:
from google.colab import files

files.download("iris-feature-store-creation-sqlite.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [3]:
!unzip iris-feature-store-creation-sqlite.zip

Archive:  iris-feature-store-creation-sqlite.zip
   creating: iris-feature-store-creation-sqlite/
  inflating: iris-feature-store-creation-sqlite/README.md  
 extracting: iris-feature-store-creation-sqlite/__init__.py  
   creating: iris-feature-store-creation-sqlite/feature_repo/
   creating: iris-feature-store-creation-sqlite/feature_repo/__pycache__/
  inflating: iris-feature-store-creation-sqlite/feature_repo/__pycache__/__init__.cpython-312.pyc  
  inflating: iris-feature-store-creation-sqlite/feature_repo/__pycache__/test_workflow.cpython-312.pyc  
  inflating: iris-feature-store-creation-sqlite/feature_repo/__pycache__/feature_definitions.cpython-312.pyc  
 extracting: iris-feature-store-creation-sqlite/feature_repo/__init__.py  
  inflating: iris-feature-store-creation-sqlite/feature_repo/feature_store.yaml  
   creating: iris-feature-store-creation-sqlite/feature_repo/data/
   creating: iris-feature-store-creation-sqlite/feature_repo/data/.ipynb_checkpoints/
  inflating: iris-

In [5]:
!pip install "numpy<2" "pandas==2.2.2"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 43.2 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
feast 0.64.0 requires numpy<3,>=2.0.0, but you have numpy 1.26.4 which is incompatible.
xarray-einstats 0.10.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
pytensor 2.38.3 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
tobler 0.14.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
jaxlib 0.7.2 requires numpy>=2.0, but you have num

In [15]:
import os
import joblib
import feast
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn import metrics

# -------------------------------------------------------
# Connect to Feast Feature Store
# -------------------------------------------------------

fs = feast.FeatureStore(repo_path="/content/iris-feature-store-creation-sqlite/feature_repo")

# -------------------------------------------------------
# Create Entity DataFrame
# -------------------------------------------------------

entity_df = pd.read_csv("/content/iris_data_adapted_for_feast.csv")

entity_df = entity_df[["iris_id", "event_timestamp"]]
entity_df["event_timestamp"] = pd.to_datetime(entity_df["event_timestamp"], utc=True)

# -------------------------------------------------------
# Retrieve historical features from Feast
# -------------------------------------------------------

training_df = fs.get_historical_features(
    entity_df=entity_df,
    features=[
        "iris_features:sepal_length",
        "iris_features:sepal_width",
        "iris_features:petal_length",
        "iris_features:petal_width",
        "iris_features:sepal_area",
        "iris_features:petal_area",
        "iris_features:sepal_ratio",
        "iris_features:petal_ratio",
        "iris_features:flower_area",
        "iris_features:petal_to_sepal_ratio",
    ],
).to_df()

# -------------------------------------------------------
# Add target labels
# -------------------------------------------------------

labels = pd.read_csv("/content/iris_data_adapted_for_feast.csv")[["iris_id", "species"]]

training_df = training_df.merge(labels, on="iris_id")

# -------------------------------------------------------
# Prepare train/test data
# -------------------------------------------------------

X = training_df.drop(
    columns=[
        "iris_id",
        "event_timestamp",
        "species",
    ]
)

y = training_df["species"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.4,
    stratify=y,
    random_state=42,
)

# -------------------------------------------------------
# Train model
# -------------------------------------------------------

model = DecisionTreeClassifier(
    max_depth=3,
    random_state=1,
)

model.fit(X_train, y_train)

# -------------------------------------------------------
# Evaluate
# -------------------------------------------------------

prediction = model.predict(X_test)

print(
    "Accuracy:",
    metrics.accuracy_score(y_test, prediction),
)

# -------------------------------------------------------
# Save predictions
# -------------------------------------------------------

results = X_test.copy()

results["Actual"] = y_test.values
results["Predicted"] = prediction

artifact_dir = "./artifacts"

os.makedirs(artifact_dir, exist_ok=True)

results.to_csv(
    f"{artifact_dir}/predictions.csv",
    index=False,
)

# -------------------------------------------------------
# Save model
# -------------------------------------------------------

joblib.dump(
    model,
    f"{artifact_dir}/model.joblib",
)

Accuracy: 1.0


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


['./artifacts/model.joblib']

In [25]:
print(X_train.columns.tolist())

['sepal_length', 'sepal_width', 'petal_length', 'petal_width', 'sepal_area', 'petal_area', 'sepal_ratio', 'petal_ratio', 'flower_area', 'petal_to_sepal_ratio']


In [3]:
entity_df.head()

,iris_id,event_timestamp
0,1001,2025-09-17 10:40:17.102131
1,1001,2025-09-18 10:40:17.102131
2,1001,2025-09-19 10:40:17.102131
3,1001,2025-09-20 10:40:17.102131
4,1001,2025-09-21 10:40:17.102131


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [6]:
entity_df.dtypes

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


,0
iris_id,int64
event_timestamp,"datetime64[ns, UTC]"


In [9]:
df = pd.read_parquet("/content/iris-feature-store-creation-sqlite/feature_repo/data/iris.parquet")

print(df.dtypes)

event_timestamp         datetime64[ns, UTC]
iris_id                               int64
sepal_length                        float64
sepal_width                         float64
petal_length                        float64
petal_width                         float64
species                              object
created_timestamp       datetime64[ns, UTC]
sepal_area                          float64
petal_area                          float64
sepal_ratio                         float64
petal_ratio                         float64
flower_area                         float64
petal_to_sepal_ratio                float64
dtype: object


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [15]:
print(df["event_timestamp"].min())
print(df["event_timestamp"].max())

2025-09-17 10:40:17.102131+00:00
2025-10-01 10:40:17.102131+00:00


In [16]:
!cd /content/iris-feature-store-creation-sqlite/feature_repo/ && feast materialize 2025-09-17T00:00:00 2025-10-02T00:00:00

/usr/local/lib/python3.12/dist-packages/matplotlib/_fontconfig_pattern.py:64: PyparsingDeprecationWarning: 'oneOf' deprecated - use 'one_of'
  prop = Group((name + Suppress("=") + comma_separated(value)) | oneOf(_CONSTANTS))
/usr/local/lib/python3.12/dist-packages/matplotlib/_fontconfig_pattern.py:85: PyparsingDeprecationWarning: 'parseString' deprecated - use 'parse_string'
  parse = parser.parseString(pattern)
/usr/local/lib/python3.12/dist-packages/matplotlib/_fontconfig_pattern.py:89: PyparsingDeprecationWarning: 'resetCache' deprecated - use 'reset_cache'
  parser.resetCache()
/usr/local/lib/python3.12/dist-packages/matplotlib/_mathtext.py:45: PyparsingDeprecationWarning: 'enablePackrat' deprecated - use 'enable_packrat'
  ParserElement.enablePackrat()
In /usr/local/lib/python3.12/dist-packages/matplotlib/mpl-data/stylelib/classic.mplstyle: 'parseString' deprecated - use 'parse_string'
In /usr/local/lib/python3.12/dist-packages/matplotlib/mpl-data/stylelib/classic.mplstyle: 'reset

In [18]:
import pandas as pd

df = pd.read_parquet("/content/iris-feature-store-creation-sqlite/feature_repo/data/iris.parquet")
print(df.shape)
print(df.head())

(45, 14)
                   event_timestamp  iris_id  sepal_length  sepal_width  \
0 2025-09-17 10:40:17.102131+00:00     1001          5.52         2.53   
1 2025-09-18 10:40:17.102131+00:00     1001          5.50         2.24   
2 2025-09-19 10:40:17.102131+00:00     1001          5.55         2.47   
3 2025-09-20 10:40:17.102131+00:00     1001          5.45         2.37   
4 2025-09-21 10:40:17.102131+00:00     1001          5.65         2.52   

   petal_length  petal_width     species                created_timestamp  \
0          3.86         1.13  versicolor 2025-10-02 10:40:17.172178+00:00   
1          3.60         1.08  versicolor 2025-10-02 10:40:17.172178+00:00   
2          3.75         1.08  versicolor 2025-10-02 10:40:17.172178+00:00   
3          3.92         1.20  versicolor 2025-10-02 10:40:17.172178+00:00   
4          3.95         1.17  versicolor 2025-10-02 10:40:17.172178+00:00   

   sepal_area  petal_area  sepal_ratio  petal_ratio  flower_area  \
0     13.9656  

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [19]:
import feast
import pandas as pd
import joblib


class IrisClassificationModel:

    def __init__(self):
        # Load trained model
        self.model = joblib.load("/content/artifacts/model.joblib")

        # Connect to Feast
        self.fs = feast.FeatureStore(repo_path="/content/iris-feature-store-creation-sqlite/feature_repo")

    def predict(self, iris_ids):

        # Fetch online features from Feast
        feature_vector = self.fs.get_online_features(
            features=[
                "iris_features:sepal_length",
                "iris_features:sepal_width",
                "iris_features:petal_length",
                "iris_features:petal_width",
                "iris_features:sepal_area",
                "iris_features:petal_area",
                "iris_features:sepal_ratio",
                "iris_features:petal_ratio",
                "iris_features:flower_area",
                "iris_features:petal_to_sepal_ratio",
            ],
            entity_rows=[
                {"iris_id": iris_id}
                for iris_id in iris_ids
            ],
        )


        df = pd.DataFrame(feature_vector.to_dict())
        # Keep only feature columns
        feature_order = [
            'sepal_length',
            'sepal_width',
            'petal_length',
            'petal_width',
            'sepal_area',
            'petal_area',
            'sepal_ratio',
            'petal_ratio',
            'flower_area',
            'petal_to_sepal_ratio'
        ]

        X = df[feature_order]
        print(df)

        # Predict species
        df["predicted_species"] = self.model.predict(X)

        return df[["iris_id", "predicted_species"]]

In [11]:
def make_iris_prediction():
    iris_ids = [1001, 1002, 1003]

    model = IrisClassificationModel()

    predictions = model.predict(iris_ids)

    print(predictions)

In [20]:
make_iris_prediction()

   iris_id  sepal_area  petal_ratio  petal_length  sepal_width  sepal_ratio  \
0     1001      12.862     3.522936          3.84         2.36     2.309322   
1     1002      14.036     6.450000          1.29         2.90     1.668965   
2     1003      16.490     4.103448          1.19         3.40     1.426471   

   petal_to_sepal_ratio  petal_area  sepal_length  petal_width  flower_area  
0              0.325424      4.1856          5.45         1.09      17.0476  
1              0.018381      0.2580          4.84         0.20      14.2940  
2              0.020928      0.3451          4.85         0.29      16.8351  
   iris_id predicted_species
0     1001        versicolor
1     1002            setosa
2     1003            setosa


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [6]:
!pwd

/content


In [9]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("/content/iris-feature-store-creation-sqlite/feature_repo/data/online_store.db")

cursor = conn.cursor()

cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")

print(cursor.fetchall())

df = pd.read_sql(
    "SELECT * FROM iris_feature_store_creation_sqlite_iris_features",
    conn,
)

print(df)

conn.close()

[('iris_feature_store_creation_sqlite_iris_features',)]
                                           entity_key          feature_name  \
0   b'\x01\x00\x00\x00\x02\x00\x00\x00\x07\x00\x00...          sepal_length   
1   b'\x01\x00\x00\x00\x02\x00\x00\x00\x07\x00\x00...           sepal_width   
2   b'\x01\x00\x00\x00\x02\x00\x00\x00\x07\x00\x00...          petal_length   
3   b'\x01\x00\x00\x00\x02\x00\x00\x00\x07\x00\x00...           petal_width   
4   b'\x01\x00\x00\x00\x02\x00\x00\x00\x07\x00\x00...            sepal_area   
5   b'\x01\x00\x00\x00\x02\x00\x00\x00\x07\x00\x00...            petal_area   
6   b'\x01\x00\x00\x00\x02\x00\x00\x00\x07\x00\x00...           sepal_ratio   
7   b'\x01\x00\x00\x00\x02\x00\x00\x00\x07\x00\x00...           petal_ratio   
8   b'\x01\x00\x00\x00\x02\x00\x00\x00\x07\x00\x00...           flower_area   
9   b'\x01\x00\x00\x00\x02\x00\x00\x00\x07\x00\x00...  petal_to_sepal_ratio   
10  b'\x01\x00\x00\x00\x02\x00\x00\x00\x07\x00\x00...          sepal_length

In [21]:
! feast init iris-feature-store-creation-bigquery

/usr/local/lib/python3.12/dist-packages/matplotlib/_fontconfig_pattern.py:64: PyparsingDeprecationWarning: 'oneOf' deprecated - use 'one_of'
  prop = Group((name + Suppress("=") + comma_separated(value)) | oneOf(_CONSTANTS))
/usr/local/lib/python3.12/dist-packages/matplotlib/_fontconfig_pattern.py:85: PyparsingDeprecationWarning: 'parseString' deprecated - use 'parse_string'
  parse = parser.parseString(pattern)
/usr/local/lib/python3.12/dist-packages/matplotlib/_fontconfig_pattern.py:89: PyparsingDeprecationWarning: 'resetCache' deprecated - use 'reset_cache'
  parser.resetCache()
/usr/local/lib/python3.12/dist-packages/matplotlib/_mathtext.py:45: PyparsingDeprecationWarning: 'enablePackrat' deprecated - use 'enable_packrat'
  ParserElement.enablePackrat()
In /usr/local/lib/python3.12/dist-packages/matplotlib/mpl-data/stylelib/classic.mplstyle: 'parseString' deprecated - use 'parse_string'
In /usr/local/lib/python3.12/dist-packages/matplotlib/mpl-data/stylelib/classic.mplstyle: 'reset